# Artificial Intelligence & Machine Learning - Task 2
## Feature Engineering, Model Optimization & Performance Comparison

**Intern | Maincrafts Technology**

---

### Objective
Build an **enhanced House Price Prediction system** that includes:
- Feature-scaled input data
- Multiple regression models
- A structured model performance comparison
- A final selected model with proper justification

### Dataset
**California Housing Dataset** (available via `scikit-learn`)
- **Target Variable:** Median House Value
- **Input Features:** Median Income, House Age, Average Rooms, Population, Location-based attributes

---
## Step 1: Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import root_mean_squared_error, r2_score

import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.float_format', '{:.4f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')

import sklearn
print("All libraries imported successfully!")
print(f"  pandas    version: {pd.__version__}")
print(f"  numpy     version: {np.__version__}")
print(f"  sklearn   version: {sklearn.__version__}")
print(f"  matplotlib version: {plt.matplotlib.__version__}")

---
## Step 2: Load the Dataset

In [ ]:
data = fetch_california_housing(as_frame=True)
df = pd.concat([data.data, data.target.rename("HousePrice")], axis=1)

print("Dataset loaded successfully!")
print(f"\nDataset Shape: {df.shape}")
print(f"\nColumns: {list(df.columns)}")
print("\nFirst 5 rows:")
df.head()

In [ ]:
print("Dataset Statistical Summary:")
df.describe()

In [ ]:
print("Missing Values Check:")
missing = df.isnull().sum()
print(missing)
print(f"\nTotal missing values: {missing.sum()}")

print("\nData Types:")
print(df.dtypes)

---
## Step 3: Separate Features and Target Variable

In [ ]:
X = df.drop("HousePrice", axis=1)
y = df["HousePrice"]

print(f"Features (X) shape: {X.shape}")
print(f"Target  (y) shape: {y.shape}")
print(f"\nFeature names: {list(X.columns)}")
print(f"\nTarget variable: HousePrice (in units of $100,000)")
print(f"  Min:  {y.min():.4f}")
print(f"  Max:  {y.max():.4f}")
print(f"  Mean: {y.mean():.4f}")

---
## Step 4: Feature Scaling (Critical Step)

> **Why Feature Scaling?**  
> Features in the California Housing dataset exist on very different numeric scales:
> - `MedInc` ranges approximately 0.5 to 15  
> - `Population` can be in the thousands  
>
> Without scaling, features with larger ranges **dominate** the model and cause unstable training.  
> `StandardScaler` normalizes each feature to **mean=0** and **std=1**.

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

print("Feature scaling applied using StandardScaler")
print("\nBefore Scaling -- Feature Statistics:")
print(X.describe().loc[['mean', 'std']].round(4))

print("\nAfter Scaling -- Feature Statistics (mean~0, std~1):")
print(X_scaled_df.describe().loc[['mean', 'std']].round(4))

In [ ]:
n_cols = 4
n_rows = (len(X.columns) + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows * 2, n_cols, figsize=(16, 4 * n_rows))
fig.suptitle('Feature Distributions: Before vs After StandardScaler',
             fontsize=14, fontweight='bold', y=1.02)

features = X.columns
colors_before = '#E74C3C'
colors_after  = '#2ECC71'

for i, feat in enumerate(features):
    row_before = (i // n_cols) * 2
    row_after  = row_before + 1
    col = i % n_cols
    axes[row_before][col].hist(X[feat], bins=30, color=colors_before, alpha=0.8, edgecolor='white')
    axes[row_before][col].set_title(f'{feat}\n(Before)', fontsize=9, color=colors_before)
    axes[row_after][col].hist(X_scaled_df[feat], bins=30, color=colors_after, alpha=0.8, edgecolor='white')
    axes[row_after][col].set_title(f'{feat}\n(After)', fontsize=9, color=colors_after)

plt.tight_layout()
plt.savefig('feature_scaling_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Feature scaling visualization saved!")

---
## Step 5: Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42
)

print("Data split completed (80% Train | 20% Test)")
print(f"  Training set: X_train={X_train.shape}, y_train={y_train.shape}")
print(f"  Testing  set: X_test={X_test.shape},  y_test={y_test.shape}")
print(f"\n  random_state=42 ensures reproducibility across runs.")

---
## Step 6: Train Multiple Models

Three regression algorithms are compared:

| Model | Rationale |
|---|---|
| **Linear Regression** | Baseline model; assumes linear relationship |
| **Ridge Regression** | Adds L2 regularization to reduce overfitting |
| **Decision Tree Regressor** | Captures non-linear relationships; depth-limited |

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "Ridge Regression": Ridge(alpha=1.0),
    "Decision Tree": DecisionTreeRegressor(max_depth=5, random_state=42)
}

print("Models defined:")
for name, model in models.items():
    print(f"  * {name}: {model}")

---
## Step 7: Model Evaluation and Comparison

In [ ]:
results = {}
predictions = {}

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    predictions[name] = y_pred

    rmse = root_mean_squared_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)

    results[name] = {
        "RMSE": rmse,
        "R2 Score": r2
    }

    print(f"  {name:25s}  RMSE={rmse:.4f}  R2={r2:.4f}")

results_df = pd.DataFrame(results).T
results_df = results_df.sort_values("RMSE")

print("\n" + "="*55)
print("  MODEL PERFORMANCE COMPARISON TABLE")
print("="*55)
print(results_df.to_string())
print("="*55)
print("\n  Lower RMSE  = Better prediction accuracy")
print("  Higher R2   = Better explanatory power")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Model Performance Comparison', fontsize=15, fontweight='bold')

model_names = list(results_df.index)
rmse_values = results_df['RMSE'].values
r2_values   = results_df['R2 Score'].values

palette = ['#2ECC71', '#3498DB', '#E74C3C']

bars1 = axes[0].bar(model_names, rmse_values, color=palette, edgecolor='black', linewidth=0.8)
axes[0].set_title('RMSE (Lower is Better)', fontsize=12, fontweight='bold')
axes[0].set_ylabel('RMSE')
axes[0].set_xlabel('Model')
for bar, val in zip(bars1, rmse_values):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[0].set_ylim(0, max(rmse_values) * 1.2)

bars2 = axes[1].bar(model_names, r2_values, color=palette, edgecolor='black', linewidth=0.8)
axes[1].set_title('R2 Score (Higher is Better)', fontsize=12, fontweight='bold')
axes[1].set_ylabel('R2 Score')
axes[1].set_xlabel('Model')
for bar, val in zip(bars2, r2_values):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
                 f'{val:.4f}', ha='center', va='bottom', fontsize=10, fontweight='bold')
axes[1].set_ylim(0, 1.1)

plt.tight_layout()
plt.savefig('model_comparison_metrics.png', dpi=150, bbox_inches='tight')
plt.show()
print("Model comparison chart saved!")

---
## Step 8: Visual Performance Validation
### Actual vs Predicted House Prices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Actual vs Predicted House Prices -- All Models', fontsize=15, fontweight='bold')

colors = ['#3498DB', '#27AE60', '#E74C3C']
model_list = list(models.keys())

for idx, name in enumerate(model_list):
    y_pred = predictions[name]
    r2_val   = results[name]['R2 Score']
    rmse_val = results[name]['RMSE']

    ax = axes[idx]
    ax.scatter(y_test, y_pred, alpha=0.3, color=colors[idx], s=15, label='Predictions')

    min_val = min(y_test.min(), y_pred.min())
    max_val = max(y_test.max(), y_pred.max())
    ax.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Perfect Fit')

    ax.set_title(f'{name}\nR2={r2_val:.4f} | RMSE={rmse_val:.4f}', fontsize=11, fontweight='bold')
    ax.set_xlabel('Actual House Prices', fontsize=10)
    ax.set_ylabel('Predicted House Prices', fontsize=10)
    ax.legend(fontsize=9)

plt.tight_layout()
plt.savefig('actual_vs_predicted.png', dpi=150, bbox_inches='tight')
plt.show()
print("Actual vs Predicted visualization saved!")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle('Residual Analysis -- Prediction Errors', fontsize=15, fontweight='bold')

for idx, name in enumerate(model_list):
    y_pred = predictions[name]
    residuals = y_test.values - y_pred

    ax = axes[idx]
    ax.scatter(y_pred, residuals, alpha=0.3, color=colors[idx], s=15)
    ax.axhline(y=0, color='red', linestyle='--', linewidth=2)
    ax.set_title(f'{name}\nResiduals vs Predicted', fontsize=11, fontweight='bold')
    ax.set_xlabel('Predicted Values', fontsize=10)
    ax.set_ylabel('Residuals (Actual - Predicted)', fontsize=10)

plt.tight_layout()
plt.savefig('residual_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print("Residual analysis plot saved!")

---
## Final Model Selection & Justification

In [ ]:
best_model_name = results_df['RMSE'].idxmin()
best_rmse = results_df.loc[best_model_name, 'RMSE']
best_r2   = results_df.loc[best_model_name, 'R2 Score']

print("=" * 60)
print("  FINAL MODEL SELECTION REPORT")
print("=" * 60)
print(f"\n  Best Performing Model: {best_model_name}")
print(f"  RMSE:     {best_rmse:.4f}")
print(f"  R2 Score: {best_r2:.4f}")
print()
print("  All Model Rankings:")
print(results_df.to_string())
print()
print("  Justification:")
print(f"  '{best_model_name}' achieves the lowest RMSE and highest R2")
print(f"  score, indicating it generalizes best to unseen data.")
print()
print("  Key Observations:")
print("  * Linear Regression: Simple baseline, performance limited by")
print("    linear assumptions on non-linear housing data.")
print("  * Ridge Regression: Adds L2 regularization; performs similarly")
print("    to Linear Regression on this well-scaled dataset.")
print("  * Decision Tree (depth=5): Captures non-linear patterns")
print("    in the data, offering improved predictive capability.")
print("=" * 60)

In [ ]:
best_model = models[best_model_name]
best_model.fit(X_train, y_train)
y_pred_best = best_model.predict(X_test)

plt.figure(figsize=(7, 6))
plt.scatter(y_test, y_pred_best, alpha=0.4, color='#2E86AB', s=15, label='Predictions')
plt.plot(
    [y_test.min(), y_test.max()],
    [y_test.min(), y_test.max()],
    color='red', linestyle='--', linewidth=2, label='Perfect Fit'
)
plt.xlabel('Actual House Prices', fontsize=12)
plt.ylabel('Predicted House Prices', fontsize=12)
plt.title(f'Best Model: {best_model_name}\nActual vs Predicted House Prices',
          fontsize=13, fontweight='bold')
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig('best_model_prediction.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nFinal model '{best_model_name}' prediction plot saved!")
print(f"  RMSE:     {root_mean_squared_error(y_test, y_pred_best):.4f}")
print(f"  R2 Score: {r2_score(y_test, y_pred_best):.4f}")

---
## Summary

| Step | Action | Status |
|------|--------|--------|
| 1 | Import Libraries | Done |
| 2 | Load California Housing Dataset | Done |
| 3 | Separate Features & Target | Done |
| 4 | Feature Scaling (StandardScaler) | Done |
| 5 | Train-Test Split (80/20, seed=42) | Done |
| 6 | Train 3 Models (Linear, Ridge, Decision Tree) | Done |
| 7 | Evaluate & Compare (RMSE, R2) | Done |
| 8 | Visualize Actual vs Predicted | Done |

---

### Key Learnings

- **Feature Scaling** is essential for distance-based and gradient-based algorithms -- it ensures all features contribute equally.
- **Ridge Regression** helps prevent overfitting through L2 regularization.
- **Decision Tree** can capture complex non-linear patterns but risks overfitting if depth is unconstrained.
- **Model selection** should be driven by measurable performance metrics (RMSE, R2), not assumptions.
- **Train-test splitting** ensures the reported performance reflects real-world generalization, not memorization.

---
*Submitted as part of Maincrafts Technology AI/ML Internship - Task 2*